In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy import stats
from sklearn.cluster import KMeans

In [ ]:
DATA_PATH   = "index_data_2026.csv"
OUTPUT_PATH = "index_data_2026_scored.csv"

# Pillar features with a 1..n rank (1 = most important); weights are derived from the ranks.
FEATURE_GROUPS = {
    "technical": {
        "features": ["cc_gb", "models_n", "datasets_n", "opus_gb", "archival_gb",
                     "bible_exists", "wiki_articles_n", "wikitongues_gb", "xeus_gb"],
        "ranks":    [1, 2, 3, 4, 5, 6, 7, 8, 9],
    },
    "internet": {
        "features": ["households_with_phones_pct", "individuals_using_internet_pct",
                     "avg_download_speed_kbps", "households_with_home_internet_pct",
                     "cybersecurity_law_exists"],
        "ranks":    [1, 2, 3, 4, 5],
    },
    "language": {
        "features": ["institutional_status", "n_speakers", "wiki_active_users",
                     "literacy_rate", "admin_1_gdp", "rnd_gdp_pct", "admin_1_hdi",
                     "admin_1_education_level"],
        "ranks":    [1, 2, 3, 4, 5, 6, 7, 8],
    },
}

BINARY_FEATURES    = ["bible_exists", "cybersecurity_law_exists"]   # 0/1 penalties
LOG_SCALE_FEATURES = ["cc_gb", "models_n", "opus_gb", "archival_gb",
                      "n_speakers", "wiki_active_users", "admin_1_gdp"]
ZERO_AS_MISSING    = ["cc_gb", "wiki_active_users"]                

# {feature: (column, value)} -> scored only where column == value, missing elsewhere.
CONDITIONAL_FEATURES = {
    "archival_gb":    ("institutional_status", 0),
    "wikitongues_gb": ("institutional_status", 0),
}

# When True: drop institutional_status from the language pillar, re-rank the rest, score it as a penalty.
INSTITUTIONAL_AS_PENALTY     = True
INSTITUTIONAL_FEATURE        = "institutional_status"
INSTITUTIONAL_PENALTY_WEIGHT = 0.25

CC_SIZE_COLUMNS = ["cc_gb"]              # parse sizes like '2.6M' / '1.5GB' into numbers
COUNT_COLUMNS   = ["wiki_articles_n"]    # coerce counts (blanks/commas) to integers

# Each entry overwrites the `set` columns on rows where (column == value).
MANUAL_CORRECTIONS = [
    {"where": ("iso_code", "unk"),                      "set": {"models_n": 2}},
    {"where": ("language_name", "Modern Hebrew"),       "set": {"institutional_status": 1}},
    {"where": ("language_name", "Telugu"),              "set": {
        "models_n": 2799, "datasets_n": 778, "cc_gb": 2600000000,
        "wiki_active_users": 374, "wiki_articles_n": 124834}},
    {"where": ("language_name", "Standard Indonesian"), "set": {"institutional_status": 1}},
]

# Give a language its country's best value on every numeric column except `exclude`.
BEST_OF_COUNTRY = [
    {"language": "Mandarin Chinese", "country": "CHN", "exclude": [
        "avg_latency_ms", "distance_to_closest_university_km",
        "distance_to_second_closest_university_km"]},
]

# Lifted for the top official language of each country.
LIFT_FEATURES = ["households_with_phones_pct", "individuals_using_internet_pct",
                 "avg_download_speed_kbps", "households_with_home_internet_pct",
                 "cybersecurity_law_exists", "literacy_rate", "admin_1_gdp",
                 "admin_1_hdi", "rnd_gdp_pct", "admin_1_education_level"]

SCORE_COLUMN_RENAME = {
    "technical":               "ai_resources",
    "internet":                "digital_infrastructure",
    "language":                "socioeconomics",
    "overall_readiness_score": "overall_score",
}
N_TIERS = 10

DROP_ON_LOAD = [
    "socioeconomic_readiness", "Overall_Readiness_Score",
    "grouped_full_index", "grouped_AI_resource_index",
    "grouped_socioeconomic_index", "grouped_infrastructural_index",
    "AI_resource_readiness", "infrastructural_readiness",
]

In [ ]:
SI_FACTORS = {'K': 1e3, 'M': 1e6, 'G': 1e9, 'T': 1e12}

def cc_to_numeric(value):
    """Parse a size like '2.6M' or '1.5GB' into a number; only a trailing SI letter counts."""
    if isinstance(value, (int, float)):
        return value
    if pd.isna(value):
        return np.nan
    s = str(value).strip().upper()
    if not s:
        return np.nan
    if s.endswith('B') and len(s) > 1 and s[-2] in SI_FACTORS:
        s = s[:-1]                      # 'GB' -> 'G'
    mult = SI_FACTORS.get(s[-1], 1.0)
    if s[-1] in SI_FACTORS:
        s = s[:-1]
    try:
        return float(s) * mult
    except ValueError:
        return np.nan


def wiki_to_numeric(value):
    """Parse a count, treating blanks as zero."""
    if pd.isna(value):
        return 0
    if isinstance(value, (int, float)):
        return int(value)
    return int(str(value).replace(',', ''))

In [ ]:
def apply_manual_corrections(df, corrections):
    """Overwrite the `set` columns on rows where (column == value)."""
    for c in corrections:
        col, val = c["where"]
        if col not in df.columns:
            continue
        mask = df[col] == val
        for target_col, target_val in c["set"].items():
            df.loc[mask, target_col] = target_val
    return df


def apply_country_ceiling(df, language, country, exclude):
    """Set each row of `language` to its country's max per numeric column, excluding `exclude`."""
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    columns_to_update = [c for c in numeric_columns if c not in exclude]
    country_max = df.loc[df["country_iso_code"] == country, columns_to_update].max()
    mask = df["language_name"] == language
    for col in columns_to_update:
        df.loc[mask, col] = country_max[col]
    return df

In [ ]:
def apply_best_of_country(data, features, target_mask, scope="all"):
    """Lift each feature on target rows to the best value across the language's countries.
    Returns (data, before-snapshot)."""
    country_best = (
        data.assign(_c=data["country_iso_code"].astype(str).str.strip())
            .groupby("_c")[features].max()
    )

    def codes_for_row(row):
        codes = []
        if pd.notna(row["country_iso_code"]):
            codes.append(str(row["country_iso_code"]).strip())
        if scope == "all" and pd.notna(row["other_country_iso_codes"]):
            codes += [c.strip() for c in str(row["other_country_iso_codes"]).split("-") if c.strip()]
        return list(dict.fromkeys(codes))

    before = data[features].copy()
    for idx in data.index[target_mask]:
        codes = [c for c in codes_for_row(data.loc[idx]) if c in country_best.index]
        if codes:
            for f in features:
                data.loc[idx, f] = country_best.loc[codes, f].max()
    return data, before


def select_top_official_per_country(df):
    """Official language (institutional_status == 1) with the most speakers per country."""
    inst = df[(df["institutional_status"] == 1) & (df["n_speakers"].notna())]
    winners = inst.groupby("country_iso_code")["n_speakers"].idxmax()
    return df.index.isin(set(winners.values))

In [ ]:
def compute_normalized_weights(features, ranks, group_name=None):
    """1..n ranks -> linearly decaying, sum-to-one weights."""
    n = len(ranks)
    if n != len(features):
        raise ValueError(f"[{group_name}] features ({len(features)}) and ranks ({n}) length mismatch")
    if n < 2:
        raise ValueError(f"[{group_name}] need at least 2 features to rank")
    if sorted(ranks) != list(range(1, n + 1)):
        raise ValueError(f"[{group_name}] ranks must be a permutation of 1..{n}, got {ranks}")
    linear_weights = [1 - (rank - 1) / (n - 1) for rank in ranks]
    total = sum(linear_weights)
    df = pd.DataFrame({"feature": features, "rank": ranks,
                       "linear_weight": linear_weights,
                       "normalized_weight": [w / total for w in linear_weights]})
    if group_name:
        df["group"] = group_name
    return df

In [ ]:
def build_weights_and_groups(feature_groups, binary_features,
                             institutional_as_penalty=True,
                             institutional_feature="institutional_status",
                             penalty_weight=0.25):
    """Build (group_definitions, weights_df, binary_features) from FEATURE_GROUPS. If
    institutional_as_penalty, move the institutional feature out of its pillar into the penalty."""
    groups = {g: {"features": list(v["features"]), "ranks": list(v["ranks"])}
              for g, v in feature_groups.items()}
    binaries = list(binary_features)

    if institutional_as_penalty:
        for g, v in groups.items():
            if institutional_feature in v["features"]:
                ordered = [f for _, f in sorted(zip(v["ranks"], v["features"]))]
                ordered = [f for f in ordered if f != institutional_feature]
                v["features"] = ordered
                v["ranks"] = list(range(1, len(ordered) + 1))

    group_definitions = {g: v["features"] for g, v in groups.items()}
    weights_df = pd.concat(
        [compute_normalized_weights(v["features"], v["ranks"], g) for g, v in groups.items()],
        ignore_index=True)

    if institutional_as_penalty:
        binaries = binaries + [institutional_feature]
        weights_df = pd.concat([weights_df, pd.DataFrame([{
            "feature": institutional_feature, "rank": np.nan,
            "linear_weight": np.nan, "normalized_weight": penalty_weight,
            "group": "penalty"}])], ignore_index=True)

    return group_definitions, weights_df, binaries

In [ ]:
def validate_inputs(df, weights_df, group_defs, binary_features, log_scale_features):
    """Raise one ValueError listing any weight/group/dtype mismatch before scoring."""
    errors = []
    for col in ("feature", "normalized_weight"):
        if col not in weights_df.columns:
            errors.append(f"weights table missing required column '{col}'")
    if errors:
        raise ValueError("Input validation failed:\n - " + "\n - ".join(errors))

    weight_features = set(weights_df["feature"])
    dups = weights_df["feature"][weights_df["feature"].duplicated()].tolist()
    if dups:
        errors.append(f"duplicate features in weights table: {dups}")
    for g, feats in group_defs.items():
        if len(feats) != len(set(feats)):
            errors.append(f"[{g}] duplicate feature(s) within group")
        for f in feats:
            if f not in df.columns:
                errors.append(f"[{g}] feature '{f}' is not a column in the dataframe")
            if f not in weight_features:
                errors.append(f"[{g}] feature '{f}' has no row in the weights table")
            if f in df.columns and f not in binary_features and not pd.api.types.is_numeric_dtype(df[f]):
                errors.append(f"[{g}] feature '{f}' is non-numeric but not listed in binary_features")
    for bf in binary_features:
        if bf not in df.columns:
            errors.append(f"binary feature '{bf}' is not a column in the dataframe")
        else:
            vals = set(pd.to_numeric(df[bf], errors="coerce").dropna().unique())
            bad = vals - {0, 1, 0.0, 1.0}
            if bad:
                errors.append(f"binary feature '{bf}' contains non-0/1 values: {sorted(bad)[:5]}")
    for f in (log_scale_features or []):
        if f not in df.columns:
            errors.append(f"log_scale feature '{f}' is not a column in the dataframe")
        elif f in binary_features:
            errors.append(f"log_scale feature '{f}' is binary; cannot log-scale")
        elif not pd.api.types.is_numeric_dtype(df[f]):
            errors.append(f"log_scale feature '{f}' is non-numeric")
    if errors:
        raise ValueError("Input validation failed:\n - " + "\n - ".join(errors))

In [ ]:
def merge_correlated_features(df, group_defs, weights_df, binary_features, log_scale_features,
                              correlation_threshold=0.85, protected_features=None, verbose=True):
    """Fold within-group feature pairs with |corr| > threshold into one weight-averaged column.
    Protected and binary features are left alone."""
    protected_features = set(protected_features or [])
    df = df.copy()
    group_defs = {g: list(v) for g, v in group_defs.items()}
    weight_map = weights_df.set_index("feature")["normalized_weight"].to_dict()

    merges = []
    for g in list(group_defs.keys()):
        feats = group_defs[g]
        cont = [f for f in feats if f not in binary_features and f not in protected_features
                and pd.api.types.is_numeric_dtype(df[f])]
        if len(cont) < 2:
            continue
        corr = df[cont].corr().abs()
        candidate_pairs = sorted(
            ((corr.loc[f1, f2], f1, f2) for f1, f2 in combinations(cont, 2)
             if corr.loc[f1, f2] > correlation_threshold),
            reverse=True, key=lambda t: t[0])
        used = set()
        for c, f1, f2 in candidate_pairs:
            if f1 in used or f2 in used:
                continue
            w1, w2 = weight_map[f1], weight_map[f2]
            if w1 + w2 == 0:
                continue
            new_name = f"{f1}__{f2}__merged"
            a, b = df[f1], df[f2]
            wa = a.notna().astype(float) * w1          # NaN-aware average: NaN only when both missing
            wb = b.notna().astype(float) * w2
            denom = wa + wb
            df[new_name] = (a.fillna(0) * wa + b.fillna(0) * wb) / denom.replace(0, np.nan)
            weight_map[new_name] = w1 + w2
            used.update((f1, f2))
            feats = [new_name if x == f1 else x for x in feats if x != f2]
            merges.append((g, f1, f2, new_name, c))
        group_defs[g] = feats

    removed = {f for _, f1, f2, _, _ in merges for f in (f1, f2)}
    merged_names = [m[3] for m in merges]
    new_weights = weights_df[~weights_df["feature"].isin(removed)].copy()
    if merged_names:
        add = pd.DataFrame({"feature": merged_names,
                            "normalized_weight": [weight_map[n] for n in merged_names]})
        new_weights = pd.concat([new_weights, add], ignore_index=True)

    new_log = []
    for f in (log_scale_features or []):
        if f in removed:
            target = next(nn for _, a, b, nn, _ in merges if f in (a, b))
            if target not in new_log:
                new_log.append(target)
        elif f not in new_log:
            new_log.append(f)

    if verbose:
        if merges:
            print(f"Merged correlated features (within-group, |corr| > {correlation_threshold:.2f}):")
            for g, f1, f2, nn, c in merges:
                print(f"  [{g}] {f1} + {f2}  (corr={c:.3f})  ->  {nn}")
        else:
            print(f"No within-group pairs exceed |corr| > {correlation_threshold}.")
    return df, group_defs, new_weights, new_log

In [ ]:
def compute_group_indices(df, final_feature_weights_df, feature_group_definitions,
                          binary_features, log_scale_features=None, epsilon=1e-6,
                          value_floor=0.01, correlation_threshold=0.85,
                          merge_correlated=True, within_group_aggregation="geometric",
                          zero_as_missing=None, feature_conditions=None,
                          language_column="language_name"):
    """Per-group indices and an overall readiness score. Continuous features are shifted,
    optionally log-scaled, min-max normalised, then aggregated (geometric); binaries penalise."""
    validate_inputs(df, final_feature_weights_df, feature_group_definitions,
                    binary_features, log_scale_features)
    if not (0 < value_floor < 1):
        raise ValueError(f"value_floor must be in (0,1), got {value_floor}")
    if within_group_aggregation not in ("arithmetic", "geometric"):
        raise ValueError("within_group_aggregation must be 'arithmetic' or 'geometric'")

    zero_as_missing = set(zero_as_missing or [])
    feature_conditions = feature_conditions or {}
    protected = zero_as_missing | set(feature_conditions)   # keep out of the merge step

    if merge_correlated:
        df_m, group_defs, weights_df, log_feats = merge_correlated_features(
            df, feature_group_definitions, final_feature_weights_df,
            binary_features, log_scale_features, correlation_threshold,
            protected_features=protected)
    else:
        df_m, group_defs = df.copy(), {g: list(v) for g, v in feature_group_definitions.items()}
        weights_df, log_feats = final_feature_weights_df.copy(), list(log_scale_features or [])

    df_processed = df_m.copy()
    weights = weights_df.copy()
    weights["normalized_weight"] = weights["normalized_weight"].replace(0, 0.01)
    weight_map = weights.set_index("feature")["normalized_weight"].to_dict()

    used_features = {f for feats in group_defs.values() for f in feats}
    continuous_features = [f for f in used_features
                           if f not in binary_features and pd.api.types.is_numeric_dtype(df_processed[f])]
    missing_aware = zero_as_missing | set(feature_conditions)

    # shift +1; missing-aware features send zeros / out-of-condition rows to NaN first
    for f in continuous_features:
        if f in missing_aware:
            s = pd.to_numeric(df_processed[f], errors="coerce")
            if f in zero_as_missing:
                s = s.mask(s == 0, np.nan)
            if f in feature_conditions:
                include = feature_conditions[f].reindex(df_processed.index).fillna(False).astype(bool)
                s = s.where(include, np.nan)
            df_processed[f] = s + 1
        else:
            df_processed[f] = pd.to_numeric(df_processed[f].fillna(0)) + 1

    for f in log_feats:
        df_processed[f] = np.log(df_processed[f] + epsilon)

    for f in continuous_features:
        lo, hi = df_processed[f].min(), df_processed[f].max()
        if hi == lo:
            df_processed.loc[df_processed[f].notna(), f] = 0.5
        else:
            df_processed[f] = ((df_processed[f] - lo) / (hi - lo)).clip(lower=value_floor)

    group_indices = pd.DataFrame(index=df.index)
    for group_name, feats in group_defs.items():
        cf = [f for f in feats if f not in binary_features and weight_map[f] > 0]
        X = df_processed[cf]
        W = np.array([weight_map[f] for f in cf])
        present = X.notna()
        den = (present * W).sum(axis=1)
        if within_group_aggregation == "arithmetic":
            num = (X.fillna(0) * W).sum(axis=1)
            group_indices[group_name] = np.where(den > 0, num / den.replace(0, np.nan), 0.5)
        else:
            num = (np.log(X).fillna(0) * W).sum(axis=1)
            group_indices[group_name] = np.where(den > 0, np.exp(num / den.replace(0, np.nan)), 1.0)

    num_groups = len(group_defs)
    s_groups = group_indices.prod(axis=1) ** (1.0 / num_groups)

    binary_penalty = pd.Series(1.0, index=df.index)
    for bf in binary_features:
        w = max(0.0, min(1.0, weight_map[bf]))
        val = pd.to_numeric(df_processed[bf], errors="coerce").fillna(0)
        binary_penalty *= np.maximum(0, 1 - w * (1 - val))

    group_indices["overall_readiness_score"] = s_groups * binary_penalty
    if language_column in df.columns:
        group_indices.insert(0, language_column, df[language_column].astype(str).str.strip())
    return group_indices.sort_values("overall_readiness_score", ascending=False)

In [ ]:
def assign_tiers(df, score_columns, n_tiers=10, random_state=42):
    """Bucket each score column into n_tiers ordered tiers (0 = best) via 1-D KMeans."""
    for col in score_columns:
        mask = df[col].notna()
        km = KMeans(n_clusters=n_tiers, random_state=random_state, n_init=10)
        labels = km.fit_predict(df.loc[mask, [col]])
        order = np.argsort(-km.cluster_centers_.flatten())
        remap = {old: new for new, old in enumerate(order)}
        df["grouped_" + col] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        df.loc[mask, "grouped_" + col] = (
            pd.Series(labels, index=df.loc[mask].index).map(remap).astype("Int64"))
    return df

In [ ]:
data = pd.read_csv(DATA_PATH)
data.columns = data.columns.str.strip()

obj_cols = data.select_dtypes(include="object").columns
data[obj_cols] = data[obj_cols].apply(lambda s: s.str.strip())

data = data.drop(columns=DROP_ON_LOAD, errors="ignore")

In [ ]:
for col in CC_SIZE_COLUMNS:
    data[col] = data[col].apply(cc_to_numeric)
for col in COUNT_COLUMNS:
    data[col] = data[col].apply(wiki_to_numeric)

data = apply_manual_corrections(data, MANUAL_CORRECTIONS)

In [ ]:
for spec in BEST_OF_COUNTRY:
    data = apply_country_ceiling(data, spec["language"], spec["country"], spec["exclude"])

In [ ]:
target_mask = select_top_official_per_country(data)
print(f"{target_mask.sum()} languages selected (one per country):")
print(data.loc[target_mask, ["language_name", "country_iso_code", "n_speakers"]]
      .sort_values("n_speakers", ascending=False).head(20).to_string(index=False))

data, before_lift = apply_best_of_country(data, LIFT_FEATURES, target_mask, scope="all")

changed = (before_lift != data[LIFT_FEATURES]) & ~(before_lift.isna() & data[LIFT_FEATURES].isna())
print(f"\n{int(changed.values.sum())} cells changed across "
      f"{int(changed.any(axis=1).sum())} languages.")

In [ ]:
feature_group_definitions, final_feature_weights_df, binary_features = build_weights_and_groups(
    FEATURE_GROUPS, BINARY_FEATURES,
    institutional_as_penalty=INSTITUTIONAL_AS_PENALTY,
    institutional_feature=INSTITUTIONAL_FEATURE,
    penalty_weight=INSTITUTIONAL_PENALTY_WEIGHT)

log_scale_features = list(LOG_SCALE_FEATURES)
zero_as_missing    = list(ZERO_AS_MISSING)
feature_conditions = {f: (data[col] == val) for f, (col, val) in CONDITIONAL_FEATURES.items()}

# Features cleaned/transformed but not in any pillar contribute nothing.
scored = {f for feats in feature_group_definitions.values() for f in feats}
suspects = {"wiki_articles_n", "datasets_n", "models_mean_downloads",
            "xeus_gb", "wikitongues_gb"} | set(log_scale_features)
orphans = sorted(f for f in suspects if f in data.columns and f not in scored
                 and f not in binary_features)
if orphans:
    print("Not in any pillar (contribute nothing):", orphans)

### Diagnostic: skew / exponential check (optional)

In [ ]:
candidate_features = [f for feats in FEATURE_GROUPS.values() for f in feats["features"]
                      if f not in binary_features]

rows = []
for f in candidate_features:
    if f not in data.columns or not np.issubdtype(data[f].dtype, np.number):
        continue
    x = data[f].dropna().astype(float)
    x = x[x >= 0]
    if len(x) < 20 or x.max() == x.min():
        continue
    rate = 1.0 / x.mean() if x.mean() > 0 else np.nan
    ks_stat, p = stats.kstest(x, "expon", args=(0, 1.0 / rate))
    rows.append({"feature": f, "skew": stats.skew(x),
                 "ks_stat": ks_stat, "p_value": p,
                 "looks_exponential": p > 0.05})

print(pd.DataFrame(rows).sort_values("skew", ascending=False).to_string(index=False))

In [ ]:
result = compute_group_indices(
    data, final_feature_weights_df, feature_group_definitions, binary_features,
    log_scale_features=log_scale_features, value_floor=0.01,
    within_group_aggregation="geometric",
    feature_conditions=feature_conditions,
    zero_as_missing=zero_as_missing)

result.head(50)

In [ ]:
raw_cols   = list(SCORE_COLUMN_RENAME)
final_cols = list(SCORE_COLUMN_RENAME.values())

data = data.drop(columns=final_cols + ["grouped_" + c for c in final_cols] + raw_cols,
                 errors="ignore")
data = data.join(result[raw_cols]).rename(columns=SCORE_COLUMN_RENAME)
data = assign_tiers(data, final_cols, n_tiers=N_TIERS)

data = data.sort_values("overall_score", ascending=False)
data

In [ ]:
data.to_csv(OUTPUT_PATH, index=False)